# Visualizing a real `inspect_evals` run with INIF

This notebook loads the output of a **real** Inspect AI evaluation and explores it with INIF. The evaluation ran the full 198 samples of `inspect_evals/gpqa_diamond` (graduate-level physics/chem/bio MCQ) against **Kimi K2.5** (Moonshot AI's 262k-context thinking MoE) via Together AI.

How the eval was produced (see `run_eval.py`):

```python
from inspect_ai import Task, eval
from inspect_ai.model import GenerateConfig
from inspect_ai.scorer import choice
from inspect_ai.solver import multiple_choice
from inspect_evals.gpqa import gpqa_diamond

# The gpqa_diamond task uses multiple_choice(cot=True) under the hood, which
# forces an explicit max_tokens=None into every generate() call and shadows
# any eval-level GenerateConfig. Rebuild the Task with a solver-level cap.
MAX_TOKENS = 260000
orig = gpqa_diamond()
task = Task(
    dataset=orig.dataset,
    solver=multiple_choice(cot=True, max_tokens=MAX_TOKENS),
    scorer=choice(),
    epochs=1,
    version=orig.version,
    metadata=orig.metadata,
)
eval(
    tasks=task,
    model="together/moonshotai/Kimi-K2.5",
    log_dir="logs",
    log_format="eval",
    max_connections=10,
    config=GenerateConfig(max_tokens=MAX_TOKENS, temperature=0.0, seed=42),
)
```

The notebook then:
1. Loads the `.eval` log into an `InifDocument` with the real Kimi K2.5 tokenizer
2. Inspects the converted structure (samples, sequences, token roles, generated spans)
3. Tags the model's final-answer choice letter (`A`/`B`/`C`/`D`)
4. Compares correct vs incorrect traces (reasoning length, answer-letter position)
5. Renders an inline HTML viewer and saves the enriched doc to disk

## 1. Load the eval log

`from_eval_file` reads the Inspect `.eval` archive, tokenizes each sample with the supplied HuggingFace tokenizer via `apply_chat_template`, deduplicates the shared system-prompt prefix into a `Sequence`, and records per-token chat roles (`system`/`user`/`assistant`/`template`) plus a `generated` annotation on the model's response — all stored as named ranges on `Sample.annotations`.

In [1]:
import glob
import warnings

warnings.filterwarnings("ignore")

from transformers import AutoTokenizer  # noqa: E402

from inif.converters.inspect_ai import from_eval_file  # noqa: E402

# Kimi ships a custom tokenizer class, so trust_remote_code=True is required.
tokenizer = AutoTokenizer.from_pretrained(
    "moonshotai/Kimi-K2.5", trust_remote_code=True
)

eval_path = sorted(glob.glob("../logs/*.eval"))[-1]
print(f"Loading: {eval_path}")

doc = from_eval_file(eval_path, tokenizer=tokenizer)

meta = doc.metadata
print(f"Model:           {meta.model.name}")
print(f"Task:            {meta.source_eval.task} (v{meta.source_eval.task_version})")
print(f"Inspect version: {meta.source_eval.framework_version}")
print(f"Eval duration:   {meta.total_time:.1f}s")
print(f"Samples:         {doc.total_samples}")
print(f"Sequences:       {len(doc.sequences)} (shared token runs)")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loading: ../logs/2026-04-24T02-23-19+00-00_task_ASeU2GcjA5DKgGAqMP3a4R.eval


Model:           together/moonshotai/Kimi-K2.5
Task:            task (v2)
Inspect version: None
Eval duration:   4323.0s
Samples:         47
Sequences:       2 (shared token runs)


## 2. Look at the converted structure

Each `Sample` stores the message texts, the tokenized chat, the eval `scores`, and the gold `target`. The GPQA system prompt + answer-format preamble is shared across every sample; INIF collapses that into a `Sequence` and stores a single sequence-reference token per sample instead of replaying the shared tokens 198 times.

In [2]:
from statistics import mean, median

print("Shared sequences (deduplicated across samples):")
for seq in doc.sequences:
    head = "".join((t.token or "") for t in seq.tokens[:6]).replace("\n", "␤")
    tail = "".join((t.token or "") for t in seq.tokens[-4:]).replace("\n", "␤")
    print(f"  {seq.id}: {seq.n_tokens:4d} tokens  —  {head!r} … {tail!r}")


def own_n(s):
    return sum(1 for t in s.tokens if not t.is_sequence_ref)


def ref_n(s):
    return sum(1 for t in s.tokens if t.is_sequence_ref)


def gen_n(s):
    # ``generated`` annotation positions are recorded against the deduplicated
    # tokens (sequence refs included), so we mask out refs before counting.
    return sum(
        1
        for pos in s.annotation_positions("generated")
        if not s.tokens[pos].is_sequence_ref
    )


def has_role_annotations(s):
    return any(ann.metadata.get("source") == "message_role" for ann in s.annotations)


own_counts = [own_n(s) for s in doc.samples]
ref_counts = [ref_n(s) for s in doc.samples]
gen_counts = [gen_n(s) for s in doc.samples]
n_with_roles = sum(1 for s in doc.samples if has_role_annotations(s))

print("\nPer-sample aggregates:")
print(
    f"  own tokens:        mean={mean(own_counts):.1f}  median={median(own_counts)}  "
    f"min={min(own_counts)}  max={max(own_counts)}"
)
print(f"  sequence refs:     mean={mean(ref_counts):.1f}  (one per shared sequence)")
print(
    f"  generated tokens:  mean={mean(gen_counts):.1f}  median={median(gen_counts)}  "
    f"min={min(gen_counts)}  max={max(gen_counts)}"
)
print(f"  samples with role annotations: {n_with_roles}/{doc.total_samples}")
print(
    f"  samples with any generated annotations: {sum(1 for g in gen_counts if g > 0)}/"
    f"{doc.total_samples}"
)

Shared sequences (deduplicated across samples):
  seq_0:   52 tokens  —  '<|im_user|>user<|im_middle|>Answer the following' … ' step before answering.␤␤'
  seq_1:    7 tokens  —  '<|im_end|><|im_assistant|>assistant<|im_middle|><think></think>' … '<|im_middle|><think></think><|im_end|>'

Per-sample aggregates:
  own tokens:        mean=10564.3  median=7987  min=1383  max=35159
  sequence refs:     mean=1.0  (one per shared sequence)
  generated tokens:  mean=10370.1  median=7814  min=1251  max=34925
  samples with role annotations: 47/47
  samples with any generated annotations: 47/47


The GPQA `choice` scorer packs three things into each score: `value` is `"C"` (correct) or `"I"` (incorrect), `answer` holds the predicted letter, and `Sample.target` carries the gold letter.

In [3]:
from collections import Counter

correct = [s for s in doc.samples if s.scores and s.scores[0].value == "C"]
incorrect = [s for s in doc.samples if s.scores and s.scores[0].value == "I"]
print(f"Correct:   {len(correct):3d} / {doc.total_samples}")
print(f"Incorrect: {len(incorrect):3d} / {doc.total_samples}")
print(f"Accuracy:  {len(correct) / doc.total_samples:.1%}")

# Gold vs predicted letter distribution.
gold_counts = Counter(s.target for s in doc.samples)
pred_counts = Counter(s.scores[0].answer for s in doc.samples if s.scores)
print("\nAnswer-letter distribution:")
print(f"  {'':8s} {'A':>4s} {'B':>4s} {'C':>4s} {'D':>4s}")
print(
    f"  {'gold':8s} "
    + " ".join(f"{gold_counts.get(letter, 0):4d}" for letter in "ABCD")
)
print(
    f"  {'pred':8s} "
    + " ".join(f"{pred_counts.get(letter, 0):4d}" for letter in "ABCD")
)

Correct:    29 / 47
Incorrect:  18 / 47
Accuracy:  61.7%

Answer-letter distribution:
              A    B    C    D
  gold        8   15   12   12
  pred        5   12   10    6


## 3. Tag the final-answer letter in each reasoning trace

GPQA prompts the model to end with `ANSWER: X`. The committed answer token lives in the generated response, so we annotate it on the deduplicated canonical document first (using the existing `generated` annotation to scope the search) and expand only afterward for position-based analysis and display. This avoids making expansion the default first step for large evals.

In [ ]:
import re

ANSWER_RE = re.compile(r"^\s*[ABCD]\s*$")


for sample in doc.samples:
    generated_positions = set(sample.annotation_positions("generated"))
    matches = [
        i
        for i, t in enumerate(sample.tokens)
        if i in generated_positions and t.token is not None and ANSWER_RE.match(t.token)
    ]
    if not matches:
        continue
    # The LAST A-D-ish token the model emitted is the committed choice.
    sample.annotate_positions("answer_letter", [matches[-1]])

# Expand after lightweight tagging so downstream position analyses see a flat stream.
doc_exp = doc.expand_sequences()

tagged_total = 0
agree_with_scorer = 0
no_tag = 0
for sample in doc_exp.samples:
    tagged = sample.select_by_annotation("answer_letter").tokens
    if not tagged:
        no_tag += 1
        continue
    tagged_total += 1
    scorer_answer = sample.scores[0].answer if sample.scores else None
    tagged_str = (tagged[0].token or "").strip()
    if scorer_answer and tagged_str == scorer_answer:
        agree_with_scorer += 1

print(f"Tagged a final letter:          {tagged_total} / {len(doc_exp.samples)}")
print(f"  agrees with Inspect scorer:   {agree_with_scorer} / {tagged_total}")
print(f"No generated A-D token found:   {no_tag}")

## 4. Correct vs incorrect: does the model reason longer when it gets it wrong?

Partition the document by score using `InifDocument.filter_samples_by_score`, then compute the mean number of `generated` tokens per group. This is the kind of light-weight, per-trace statistic the INIF format is designed to make trivial.

In [ ]:
from statistics import mean


def generated_token_count(sample) -> int:
    return len(sample.annotation_positions("generated"))


correct = doc_exp.filter_samples_by_score("choice", lambda v: v == "C")
incorrect = doc_exp.filter_samples_by_score("choice", lambda v: v == "I")

print(
    f"Correct   (n={len(correct):2d}):  mean generated tokens = "
    f"{mean(generated_token_count(s) for s in correct):.1f}"
)
print(
    f"Incorrect (n={len(incorrect):2d}):  mean generated tokens = "
    f"{mean(generated_token_count(s) for s in incorrect):.1f}"
)

## 5. Where does each answer letter appear in the trace?

For each sample, compute the relative position (0–1) of the committed answer letter within its generated segment. A late position means the model stuck to a first-line draft style; an early position suggests it rushed to commit before reasoning.

In [6]:
def answer_relative_position(sample) -> float | None:
    gen_positions = sample.annotation_positions("generated")
    ans_positions = sample.annotation_positions("answer_letter")
    if not gen_positions or not ans_positions:
        return None
    gen_start, gen_end = gen_positions[0], gen_positions[-1]
    gen_len = gen_end - gen_start + 1
    return (ans_positions[0] - gen_start) / max(gen_len - 1, 1)


positions_correct = [
    answer_relative_position(s)
    for s in doc_exp.samples
    if s.scores and s.scores[0].value == "C"
]
positions_incorrect = [
    answer_relative_position(s)
    for s in doc_exp.samples
    if s.scores and s.scores[0].value == "I"
]
positions_correct = [p for p in positions_correct if p is not None]
positions_incorrect = [p for p in positions_incorrect if p is not None]


def bucketize(positions, bins=(0.0, 0.25, 0.5, 0.75, 1.001)):
    counts = [0] * (len(bins) - 1)
    for p in positions:
        for i in range(len(bins) - 1):
            if bins[i] <= p < bins[i + 1]:
                counts[i] += 1
                break
    return counts


labels = ["0.00–0.25", "0.25–0.50", "0.50–0.75", "0.75–1.00"]
print(f"{'bucket':15s} {'correct':>8s} {'incorrect':>10s}")
cc = bucketize(positions_correct)
ci = bucketize(positions_incorrect)
for label, a, b in zip(labels, cc, ci):
    print(f"  {label:13s} {a:8d} {b:10d}")
print(
    f"  {'mean':13s} "
    f"{mean(positions_correct) if positions_correct else float('nan'):8.3f} "
    f"{mean(positions_incorrect) if positions_incorrect else float('nan'):10.3f}"
)

bucket           correct  incorrect
  0.00–0.25            0          0
  0.25–0.50            0          0
  0.50–0.75            0          0
  0.75–1.00           29         18
  mean             0.999      0.997


## 6. Inline visualization with the INIF viewer

`InifDocument.show` renders the document as colored HTML in the notebook: tokens are colored by chat-template role, and annotations (`generated`, `answer_letter`, ...) are highlighted on hover. Let's look at the first correct and the first incorrect sample side-by-side as separate sub-documents.

In [ ]:
correct_id = correct[0].id
incorrect_id = incorrect[0].id

correct_doc = doc_exp.subset(lambda s: s.id == correct_id)
print(f"Correct sample: {correct_id}")
correct_doc.show()

In [ ]:
incorrect_doc = doc_exp.subset(lambda s: s.id == incorrect_id)
print(f"Incorrect sample: {incorrect_id}")
incorrect_doc.show()

## 7. Persist the enriched document

Save the deduplicated document with its `answer_letter` annotations back into INIF format for downstream sharing. Annotations live on the `Sample` (not on individual tokens), so they survive serialisation as-is — we keep the compact sequence-reference form, save it, and round-trip it.

In [ ]:
from inif import InifDocument, validate

doc.save("gpqa_analysis.inif.json")
doc.save("gpqa_analysis.inif")  # indexed archive

validate(doc.to_dict())

reloaded = InifDocument.load("gpqa_analysis.inif")
n_answer_annotations = sum(
    len(s.annotation_positions("answer_letter")) for s in reloaded.samples
)
print(
    "Saved, reloaded, validated.  "
    f"answer_letter annotations round-tripped: {n_answer_annotations}"
)